# SHL Hiring Assessment 2026 — Grammar Scoring Engine for Spoken English

**Task.** Given a 45–60 s `.wav` recording of a speaker, predict a continuous *grammar score* in **[0, 5]**
that matches the MOS-Likert rubric used by human raters (1 = very limited control of structure … 5 = accurate,
adept control of complex grammar).

**Data.** 769 labelled training clips, 216 unlabelled test clips. Leaderboard metrics: **Pearson correlation** and **RMSE**.

---

## 1. Report (summary of the approach)

### 1.1 Why a transcript-first pipeline
Grammar is a property of the *words* a speaker produces, not of the waveform. A model that works directly on
spectrograms would have to learn speech recognition *and* grammatical judgement from only 769 samples, which is
hopeless. Instead the pipeline is:

```
.wav ──► Whisper ASR ──► transcript (+ word timings, confidence)
                               │
                 ┌─────────────┼───────────────────────────┐
                 ▼             ▼                           ▼
        grammar-error      fluency / syntactic        sentence-embedding
        features           / lexical features         (semantic quality)
        (LanguageTool)     (spaCy, timings)           (MiniLM / DeBERTa)
                 └─────────────┼───────────────────────────┘
                               ▼
                  Ridge  +  LightGBM  (+ optional DeBERTa fine-tune)
                               ▼
                 OOF-weighted ensemble → clip to [0, 5]
```

### 1.2 Preprocessing
* Audio is loaded mono at 16 kHz (Whisper's native rate). Clips are **not** trimmed or normalised — leading
  silence and low energy are themselves useful signals (see the 0-score clips below).
* Acoustic descriptors computed per clip: duration, RMS energy, fraction of silent frames, number/length of pauses.
* **Whisper** (`faster-whisper`, *medium* by default; set `WHISPER_MODEL = "large-v3"` for best quality on a GPU)
  transcribes each clip with word-level timestamps. We keep `avg_logprob`, `no_speech_prob`, and the fraction of
  words with low ASR confidence — low ASR confidence correlates with disfluent, hard-to-parse speech.
* Transcripts are cached to `transcripts.csv` so the expensive step runs once.

### 1.3 Feature families
| Family | Examples | Why it matters for the rubric |
|---|---|---|
| **Grammar errors** (LanguageTool) | errors / 100 words, counts by category (grammar, typos, style) | Directly measures the rubric's "makes basic grammatical mistakes" |
| **Syntax** (spaCy) | mean dependency-tree depth, subordinate-clause rate, sentence-fragment rate, POS ratios | Rubric levels 4–5 are about *complex structures* and *complete sentences* |
| **Fluency** (timings) | words / min, pause count, mean pause length, filler-word rate, repetition rate | Hesitation and restarts accompany poor control of structure |
| **Lexical** | type–token ratio, mean word length, long-word ratio | Proxy for proficiency; correlated with grammar scores |
| **ASR confidence** | `avg_logprob`, `no_speech_prob`, low-confidence word fraction | Detects empty / unintelligible clips (the 0-score cases) |
| **Semantic embedding** | 384-d MiniLM sentence embedding of the transcript | Captures overall "well-formedness" that hand-crafted counts miss |

### 1.4 Models and validation
* **5-fold KFold** (seed 42). All metrics reported out-of-fold (OOF) — this is the honest estimate of test performance.
* **Model A — Ridge** on standardised handcrafted features + embedding.
* **Model B — LightGBM** on handcrafted features only (interpretable; gives feature importances).
* **Model C — DeBERTa-v3-base regression head** fine-tuned on raw transcripts (optional, GPU; `RUN_DEBERTA = True`).
* Final prediction = non-negative least-squares blend of the OOF predictions, clipped to [0, 5].

### 1.5 Evaluation
RMSE, Pearson *r*, Spearman *ρ* are reported both **out-of-fold** and **in-sample on the full training set**
(the latter is required by the assessment). Visualisations: label distribution, duration distribution, feature–label
correlations, predicted-vs-actual, residuals by score band, LightGBM feature importance.

> **Results are filled in automatically by the cells at the end of the notebook** (Section 9).

## 2. Environment & configuration

In [ ]:
# ---- Installs (idempotent; comment out if the environment already has them) ----
import importlib, subprocess, sys

def _pip(*pkgs):
    missing = [p for p in pkgs if importlib.util.find_spec(p.split("==")[0].replace("-", "_")) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_pip("faster-whisper", "librosa", "soundfile", "lightgbm", "sentence-transformers",
     "language-tool-python", "spacy", "seaborn", "tqdm")
# spaCy English model
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)

In [ ]:
import os, re, json, math, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from scipy.optimize import nnls

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED)

# ---------------- Paths: auto-detect Kaggle vs local layout ----------------
CANDIDATES = [Path("/kaggle/input"), Path("./data"), Path(".")]
def _find(name):
    for root in CANDIDATES:
        hits = list(root.rglob(name)) if root.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

TRAIN_CSV = _find("train.csv")
TEST_CSV  = _find("test.csv")
# audio folders: common layouts are audios/train, audios/test, or a single folder
def _audio_dir(split):
    for root in CANDIDATES:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_dir() and p.name.lower() in {split, f"{split}_audio", f"audios_{split}"} and any(p.glob("*.wav")):
                return p
    # fallback: single folder containing all wavs
    for root in CANDIDATES:
        if root.exists():
            for p in root.rglob("*.wav"):
                return p.parent
    raise FileNotFoundError(f"audio dir for {split}")

TRAIN_AUDIO = _audio_dir("train")
TEST_AUDIO  = _audio_dir("test")
WORK = Path("./work"); WORK.mkdir(exist_ok=True)

# ---------------- Switches ----------------
WHISPER_MODEL = "medium"        # "large-v3" for best accuracy on GPU, "small" for a quick run
RUN_DEBERTA   = False           # set True on a GPU for the strongest single model
N_FOLDS       = 5

print("train csv :", TRAIN_CSV)
print("test  csv :", TEST_CSV)
print("train wav :", TRAIN_AUDIO)
print("test  wav :", TEST_AUDIO)

## 3. Data loading & exploratory analysis

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
train["path"] = train.filename.map(lambda f: str(TRAIN_AUDIO / f))
test["path"]  = test.filename.map(lambda f: str(TEST_AUDIO / f))

print(train.shape, test.shape)
print("missing train wavs:", (~train.path.map(os.path.exists)).sum())
print("missing test  wavs:", (~test.path.map(os.path.exists)).sum())
train.label.describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = sorted(train.label.unique())
sns.countplot(x="label", data=train, order=order, ax=ax[0], color="#4C72B0")
ax[0].set_title("Label distribution (train)"); ax[0].set_xlabel("Grammar score")
sns.histplot(train.label, bins=11, kde=True, ax=ax[1], color="#4C72B0")
ax[1].set_title("Density"); ax[1].set_xlabel("Grammar score")
plt.tight_layout(); plt.show()

print(train.label.value_counts().sort_index().to_string())

**Observations.**
* Scores are given in 0.5 steps; the median is 3.0 and the distribution is roughly bell-shaped between 2 and 5.
* **37 clips carry a score of 0.0** — outside the 1–5 rubric. Their filenames (`audio_50xx`) sit in a separate ID range
  from the rest (`audio_0 … audio_784`), so they are almost certainly a distinct batch: empty, silent, or
  non-English recordings. They matter a lot for RMSE (a 0 predicted as 3 costs 9 squared-error units), so the pipeline
  keeps ASR-confidence / speech-ratio features specifically to catch them. We check this hypothesis after transcription.

## 4. Audio preprocessing & acoustic descriptors

In [ ]:
import librosa

SR = 16000

def acoustic_features(path, sr=SR, frame=0.025, hop=0.010, silence_db=-40):
    '''Load mono 16 kHz audio and compute energy / silence / pause descriptors.'''
    y, _ = librosa.load(path, sr=sr, mono=True)
    dur = len(y) / sr
    if dur == 0:
        return dict(duration=0, rms_mean=0, rms_std=0, silence_ratio=1.0,
                    n_pauses=0, mean_pause=0, max_pause=0, leading_silence=0)
    rms = librosa.feature.rms(y=y, frame_length=int(frame*sr), hop_length=int(hop*sr))[0]
    db  = librosa.amplitude_to_db(rms + 1e-9, ref=np.max)
    voiced = db > silence_db
    # pauses = runs of unvoiced frames >= 0.3 s
    runs, cur = [], 0
    for v in voiced:
        if not v: cur += 1
        else:
            if cur: runs.append(cur); cur = 0
    if cur: runs.append(cur)
    pauses = np.array([r*hop for r in runs if r*hop >= 0.3])
    lead = 0
    for v in voiced:
        if v: break
        lead += 1
    return dict(duration=dur,
                rms_mean=float(rms.mean()), rms_std=float(rms.std()),
                silence_ratio=float(1 - voiced.mean()),
                n_pauses=len(pauses),
                mean_pause=float(pauses.mean()) if len(pauses) else 0.0,
                max_pause=float(pauses.max()) if len(pauses) else 0.0,
                leading_silence=lead*hop)

def run_cached(df, fn, cache, desc):
    '''Apply fn(path) row-wise, caching the result dataframe on disk.'''
    cache = WORK / cache
    if cache.exists():
        out = pd.read_csv(cache)
        if len(out) == len(df) and (out.filename.values == df.filename.values).all():
            return out
    rows = [dict(filename=f, **fn(p)) for f, p in tqdm(zip(df.filename, df.path), total=len(df), desc=desc)]
    out = pd.DataFrame(rows); out.to_csv(cache, index=False)
    return out

ac_train = run_cached(train, acoustic_features, "acoustic_train.csv", "acoustic train")
ac_test  = run_cached(test,  acoustic_features, "acoustic_test.csv",  "acoustic test")
ac_train.describe().T

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(ac_train.duration, bins=30, ax=ax[0]); ax[0].set_title("Clip duration (s)")
tmp = ac_train.merge(train[["filename", "label"]], on="filename")
sns.boxplot(x="label", y="silence_ratio", data=tmp, ax=ax[1]); ax[1].set_title("Silence ratio by score")
sns.boxplot(x="label", y="n_pauses", data=tmp, ax=ax[2]); ax[2].set_title("Pauses (≥0.3 s) by score")
plt.tight_layout(); plt.show()

## 5. Speech-to-text with Whisper

In [ ]:
import torch
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
print("device:", DEVICE, "| model:", WHISPER_MODEL)

try:
    from faster_whisper import WhisperModel
    _whisper = WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type=COMPUTE)
    BACKEND = "faster-whisper"
except Exception as e:           # fallback to openai-whisper
    print("faster-whisper unavailable:", e)
    import whisper
    _whisper = whisper.load_model(WHISPER_MODEL, device=DEVICE)
    BACKEND = "openai-whisper"
print("backend:", BACKEND)

def transcribe(path):
    '''Return transcript text + ASR confidence + word-level timing stats.'''
    words, logprobs, nsp = [], [], []
    if BACKEND == "faster-whisper":
        segs, info = _whisper.transcribe(path, language="en", beam_size=5, word_timestamps=True,
                                         vad_filter=False, condition_on_previous_text=False)
        for s in segs:
            logprobs.append(s.avg_logprob); nsp.append(s.no_speech_prob)
            for w in (s.words or []):
                words.append((w.word.strip(), w.start, w.end, w.probability))
    else:
        res = _whisper.transcribe(path, language="en", word_timestamps=True, fp16=(DEVICE == "cuda"))
        for s in res["segments"]:
            logprobs.append(s["avg_logprob"]); nsp.append(s["no_speech_prob"])
            for w in s.get("words", []):
                words.append((w["word"].strip(), w["start"], w["end"], w.get("probability", 1.0)))
    text = " ".join(w[0] for w in words).strip()
    text = re.sub(r"\s+([,.!?])", r"\1", text)
    starts = np.array([w[1] for w in words]); ends = np.array([w[2] for w in words])
    probs  = np.array([w[3] for w in words]) if words else np.array([1.0])
    gaps   = (starts[1:] - ends[:-1]) if len(words) > 1 else np.array([0.0])
    speech_span = float(ends[-1] - starts[0]) if len(words) else 0.0
    return dict(
        text=text,
        n_words_asr=len(words),
        avg_logprob=float(np.mean(logprobs)) if logprobs else -5.0,
        no_speech_prob=float(np.mean(nsp)) if nsp else 1.0,
        low_conf_frac=float((probs < 0.5).mean()),
        word_prob_mean=float(probs.mean()),
        speech_span=speech_span,
        wpm=len(words) / max(speech_span, 1e-3) * 60,
        gap_mean=float(gaps.mean()), gap_max=float(gaps.max()),
        n_long_gaps=int((gaps > 0.5).sum()),
    )

tr_asr = run_cached(train, transcribe, "transcripts_train.csv", "whisper train")
te_asr = run_cached(test,  transcribe, "transcripts_test.csv",  "whisper test")
tr_asr["text"] = tr_asr.text.fillna(""); te_asr["text"] = te_asr.text.fillna("")
tr_asr.head()

In [ ]:
# ---- Sanity check the 0-score hypothesis ----
chk = tr_asr.merge(train[["filename", "label"]], on="filename")
zero = chk[chk.label == 0]
print("0-score clips:", len(zero))
print(zero[["filename", "n_words_asr", "no_speech_prob", "avg_logprob"]].describe().T[["mean", "min", "max"]])
print("\nExample transcripts of 0-score clips:")
for t in zero.text.head(5): print(" •", (t[:120] + "…") if len(t) > 120 else t or "<empty>")
print("\nExample transcripts by score:")
for s in [2.0, 3.0, 4.0, 5.0]:
    t = chk[chk.label == s].text.iloc[0]
    print(f"[{s}]", t[:200], "…")

## 6. Feature engineering from transcripts

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

# ---- LanguageTool (needs Java). If unavailable, grammar-error features are zeros and we rely on the rest. ----
try:
    import language_tool_python
    _lt = language_tool_python.LanguageTool("en-US")
    HAVE_LT = True
except Exception as e:
    print("LanguageTool unavailable → grammar-error features disabled:", e)
    HAVE_LT = False

FILLERS = {"um", "uh", "hmm", "mm", "ah", "er", "like", "you know", "actually", "basically", "so"}
LT_CATS = ["GRAMMAR", "TYPOS", "STYLE", "PUNCTUATION", "MISC", "CONFUSED_WORDS", "REDUNDANCY",
           "TYPOGRAPHY", "CASING", "COLLOCATIONS"]

def tree_depth(tok):
    '''Distance from a token to the root of its dependency tree (spaCy roots have head.i == i).'''
    d, n = 0, tok
    while n.head.i != n.i and d < 200:
        n = n.head; d += 1
    return d

def text_features(text):
    f = {}
    words = re.findall(r"[A-Za-z']+", text.lower())
    nw = len(words)
    f["n_words"] = nw
    f["n_chars"] = len(text)
    f["ttr"] = len(set(words)) / nw if nw else 0
    f["mean_word_len"] = np.mean([len(w) for w in words]) if nw else 0
    f["long_word_ratio"] = np.mean([len(w) >= 7 for w in words]) if nw else 0
    f["filler_rate"] = sum(w in FILLERS for w in words) / nw * 100 if nw else 0
    # immediate repetitions ("the the", "I I") → restarts
    f["repeat_rate"] = sum(a == b for a, b in zip(words, words[1:])) / nw * 100 if nw else 0

    doc = nlp(text) if text.strip() else nlp("")
    sents = [s for s in doc.sents if len(s.text.strip()) > 0]
    ns = len(sents)
    f["n_sents"] = ns
    f["mean_sent_len"] = nw / ns if ns else 0
    f["sent_len_std"] = np.std([len(s) for s in sents]) if ns else 0
    f["frag_rate"] = np.mean([not any(t.dep_ == "ROOT" and t.pos_ in {"VERB", "AUX"} for t in s) for s in sents]) if ns else 1.0
    toks = [t for t in doc if not t.is_space and not t.is_punct]
    nt = len(toks)
    f["dep_depth_mean"] = np.mean([tree_depth(t) for t in toks]) if nt else 0
    f["dep_depth_max"] = max([tree_depth(t) for t in toks], default=0)
    f["subord_rate"] = sum(t.dep_ in {"advcl", "ccomp", "xcomp", "acl", "relcl", "csubj"} for t in toks) / ns if ns else 0
    f["conj_rate"]   = sum(t.dep_ == "conj" for t in toks) / ns if ns else 0
    for pos in ["NOUN", "VERB", "ADJ", "ADV", "PRON", "ADP", "DET", "AUX", "SCONJ", "CCONJ"]:
        f[f"pos_{pos}"] = sum(t.pos_ == pos for t in toks) / nt if nt else 0
    f["past_tense_rate"] = sum("Tense=Past" in t.morph for t in toks) / nt if nt else 0

    if HAVE_LT and text.strip():
        try:
            matches = _lt.check(text)
        except Exception:
            matches = []
        f["lt_errors"] = len(matches)
        f["lt_errors_per100w"] = len(matches) / nw * 100 if nw else 0
        cats = Counter(m.category for m in matches)
        for c in LT_CATS:
            f[f"lt_{c.lower()}"] = cats.get(c, 0) / nw * 100 if nw else 0
        f["lt_grammar_only_per100w"] = (cats.get("GRAMMAR", 0) + cats.get("CONFUSED_WORDS", 0)) / nw * 100 if nw else 0
    else:
        for k in ["lt_errors", "lt_errors_per100w", "lt_grammar_only_per100w"] + [f"lt_{c.lower()}" for c in LT_CATS]:
            f[k] = 0.0
    return f

def build_text_feats(asr_df, cache):
    cache = WORK / cache
    if cache.exists():
        out = pd.read_csv(cache)
        if (out.filename.values == asr_df.filename.values).all():
            return out
    rows = [dict(filename=fn, **text_features(t)) for fn, t in tqdm(zip(asr_df.filename, asr_df.text), total=len(asr_df))]
    out = pd.DataFrame(rows); out.to_csv(cache, index=False); return out

tf_train = build_text_feats(tr_asr, "textfeat_train.csv")
tf_test  = build_text_feats(te_asr, "textfeat_test.csv")
tf_train.describe().T.head(15)

In [ ]:
# ---- Sentence embeddings (semantic / well-formedness signal) ----
from sentence_transformers import SentenceTransformer
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
_st = SentenceTransformer(EMB_MODEL, device=DEVICE)

def embed(texts, cache):
    cache = WORK / cache
    if cache.exists():
        return np.load(cache)
    E = _st.encode(list(texts), batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache, E); return E

E_train = embed(tr_asr.text.fillna(""), "emb_train.npy")
E_test  = embed(te_asr.text.fillna(""), "emb_test.npy")
E_train.shape, E_test.shape

In [ ]:
# ---- Assemble the design matrices ----
ASR_COLS = ["n_words_asr", "avg_logprob", "no_speech_prob", "low_conf_frac", "word_prob_mean",
            "speech_span", "wpm", "gap_mean", "gap_max", "n_long_gaps"]

def assemble(ac, asr, tf):
    X = ac.merge(asr[["filename"] + ASR_COLS], on="filename").merge(tf, on="filename")
    X["speech_ratio"] = X.speech_span / X.duration.clip(lower=1e-3)
    X["words_per_sec_audio"] = X.n_words / X.duration.clip(lower=1e-3)
    return X.set_index("filename")

Xh_train = assemble(ac_train, tr_asr, tf_train).loc[train.filename]
Xh_test  = assemble(ac_test,  te_asr, tf_test).loc[test.filename]
FEATS = list(Xh_train.columns)
y = train.label.values.astype(float)
print(len(FEATS), "handcrafted features")

In [ ]:
# ---- Which handcrafted features move with the label? ----
corr = Xh_train.apply(lambda c: pearsonr(c.fillna(0), y)[0] if c.std() > 0 else 0).sort_values()
top = pd.concat([corr.head(12), corr.tail(12)])
plt.figure(figsize=(8, 7))
sns.barplot(x=top.values, y=top.index, palette=["#C44E52" if v < 0 else "#4C72B0" for v in top.values])
plt.title("Pearson correlation of handcrafted features with grammar score"); plt.xlabel("r")
plt.axvline(0, color="k", lw=.8); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
tmp = Xh_train.copy(); tmp["label"] = y
sns.boxplot(x="label", y="lt_errors_per100w", data=tmp, ax=ax[0]); ax[0].set_title("Grammar errors / 100 words")
sns.boxplot(x="label", y="dep_depth_mean", data=tmp, ax=ax[1]);     ax[1].set_title("Mean dependency depth")
sns.boxplot(x="label", y="wpm", data=tmp, ax=ax[2]);                ax[2].set_title("Words per minute")
plt.tight_layout(); plt.show()

## 7. Modelling with 5-fold cross-validation

In [ ]:
def metrics(y_true, y_pred):
    return dict(RMSE=float(np.sqrt(mean_squared_error(y_true, y_pred))),
                Pearson=float(pearsonr(y_true, y_pred)[0]),
                Spearman=float(spearmanr(y_true, y_pred)[0]))

def report(name, y_true, y_pred):
    m = metrics(y_true, y_pred)
    print(f"{name:<32} RMSE={m['RMSE']:.4f}  Pearson={m['Pearson']:.4f}  Spearman={m['Spearman']:.4f}")
    return m

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(Xh_train))
oof, test_pred, insample = {}, {}, {}

In [ ]:
# ---- Model A: Ridge on handcrafted + embedding ----
XA_train = np.hstack([Xh_train.fillna(0).values, E_train])
XA_test  = np.hstack([Xh_test.fillna(0).values,  E_test])

def ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=30.0))

oof["ridge"] = np.zeros(len(y)); test_pred["ridge"] = np.zeros(len(test))
for tr_i, va_i in FOLDS:
    m = ridge().fit(XA_train[tr_i], y[tr_i])
    oof["ridge"][va_i] = m.predict(XA_train[va_i])
    test_pred["ridge"] += m.predict(XA_test) / N_FOLDS
insample["ridge"] = ridge().fit(XA_train, y).predict(XA_train)
_ = report("Ridge (feats+emb) OOF", y, np.clip(oof["ridge"], 0, 5))

In [ ]:
# ---- Model B: LightGBM on handcrafted features ----
import lightgbm as lgb
LGB_PARAMS = dict(objective="regression", learning_rate=0.03, num_leaves=15, min_child_samples=15,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                  n_estimators=2000, random_state=SEED, verbose=-1)

oof["lgb"] = np.zeros(len(y)); test_pred["lgb"] = np.zeros(len(test)); imp = np.zeros(len(FEATS)); best_iters = []
for tr_i, va_i in FOLDS:
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(Xh_train.iloc[tr_i], y[tr_i], eval_set=[(Xh_train.iloc[va_i], y[va_i])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof["lgb"][va_i] = m.predict(Xh_train.iloc[va_i])
    test_pred["lgb"] += m.predict(Xh_test) / N_FOLDS
    imp += m.booster_.feature_importance("gain") / N_FOLDS
    best_iters.append(m.best_iteration_ or LGB_PARAMS["n_estimators"])
# full-data fit for the (required) in-sample metric, sized by the mean early-stopped iteration count
m_full = lgb.LGBMRegressor(**{**LGB_PARAMS, "n_estimators": int(np.mean(best_iters))}).fit(Xh_train, y)
insample["lgb"] = m_full.predict(Xh_train)
_ = report("LightGBM (feats) OOF", y, np.clip(oof["lgb"], 0, 5))

imp_s = pd.Series(imp, index=FEATS).sort_values(ascending=False).head(20)
plt.figure(figsize=(8, 6)); sns.barplot(x=imp_s.values, y=imp_s.index, color="#4C72B0")
plt.title("LightGBM feature importance (gain, mean over folds)"); plt.tight_layout(); plt.show()

In [ ]:
# ---- Model C (optional, GPU): DeBERTa-v3-base regression fine-tune on transcripts ----
if RUN_DEBERTA:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    from torch.utils.data import DataLoader, Dataset

    DEB = "microsoft/deberta-v3-base"; MAXLEN = 384; EPOCHS = 5; LR = 2e-5; BS = 8
    tok = AutoTokenizer.from_pretrained(DEB)

    class TDS(Dataset):
        def __init__(self, texts, ys=None): self.t, self.y = list(texts), ys
        def __len__(self): return len(self.t)
        def __getitem__(self, i):
            enc = tok(self.t[i], truncation=True, max_length=MAXLEN, padding="max_length", return_tensors="pt")
            item = {k: v.squeeze(0) for k, v in enc.items()}
            if self.y is not None: item["labels"] = torch.tensor(self.y[i] / 5.0, dtype=torch.float)
            return item

    def predict(model, texts):
        model.eval(); out = []
        with torch.no_grad():
            for b in DataLoader(TDS(texts), batch_size=32):
                b = {k: v.to(DEVICE) for k, v in b.items()}
                out.append(model(**b).logits.squeeze(-1).float().cpu().numpy())
        return np.concatenate(out) * 5.0

    def train_fold(tr_texts, tr_y, va_texts=None):
        model = AutoModelForSequenceClassification.from_pretrained(DEB, num_labels=1, problem_type="regression").to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        dl = DataLoader(TDS(tr_texts, tr_y), batch_size=BS, shuffle=True)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * EPOCHS * len(dl)), EPOCHS * len(dl))
        for ep in range(EPOCHS):
            model.train()
            for b in dl:
                b = {k: v.to(DEVICE) for k, v in b.items()}
                loss = model(**b).loss; loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad()
        return model

    texts_tr, texts_te = tr_asr.text.fillna("").values, te_asr.text.fillna("").values
    oof["deberta"] = np.zeros(len(y)); test_pred["deberta"] = np.zeros(len(test))
    for k, (tr_i, va_i) in enumerate(FOLDS):
        mdl = train_fold(texts_tr[tr_i], y[tr_i])
        oof["deberta"][va_i] = predict(mdl, texts_tr[va_i])
        test_pred["deberta"] += predict(mdl, texts_te) / N_FOLDS
        del mdl; torch.cuda.empty_cache()
    mdl = train_fold(texts_tr, y); insample["deberta"] = predict(mdl, texts_tr); del mdl
    _ = report("DeBERTa-v3 OOF", y, np.clip(oof["deberta"], 0, 5))
else:
    print("RUN_DEBERTA = False → skipping transformer fine-tune.")

In [ ]:
# ---- Ensemble: non-negative least squares on OOF predictions ----
names = list(oof)
O = np.column_stack([oof[n] for n in names])
w, _ = nnls(O, y); w = w / w.sum()
print("blend weights:", dict(zip(names, np.round(w, 3))))

oof_blend  = np.clip(O @ w, 0, 5)
test_blend = np.clip(np.column_stack([test_pred[n] for n in names]) @ w, 0, 5)
ins_blend  = np.clip(np.column_stack([insample[n] for n in names]) @ w, 0, 5)

print()
res = pd.DataFrame({n: metrics(y, np.clip(oof[n], 0, 5)) for n in names}).T
res.loc["ENSEMBLE (OOF)"] = metrics(y, oof_blend)
res

## 8. Evaluation & diagnostics

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
# predicted vs actual
sns.scatterplot(x=y + np.random.uniform(-.08, .08, len(y)), y=oof_blend, alpha=.5, ax=ax[0])
ax[0].plot([0, 5], [0, 5], "r--"); ax[0].set_xlabel("True score"); ax[0].set_ylabel("OOF prediction")
ax[0].set_title(f"OOF: r={pearsonr(y, oof_blend)[0]:.3f}, RMSE={np.sqrt(mean_squared_error(y, oof_blend)):.3f}")
# residuals by band
resid = pd.DataFrame({"label": y, "residual": oof_blend - y})
sns.boxplot(x="label", y="residual", data=resid, ax=ax[1]); ax[1].axhline(0, color="k", lw=.8)
ax[1].set_title("Residual (pred − true) by true score")
# distributions
sns.kdeplot(y, label="true", ax=ax[2]); sns.kdeplot(oof_blend, label="OOF pred", ax=ax[2]); sns.kdeplot(test_blend, label="test pred", ax=ax[2])
ax[2].legend(); ax[2].set_title("Score distributions")
plt.tight_layout(); plt.show()

# per-band RMSE: where does the model struggle?
band = resid.groupby("label").residual.apply(lambda r: np.sqrt((r**2).mean())).rename("RMSE")
print(band.round(3).to_string())

In [ ]:
# ---- Worst OOF errors: read the transcripts to understand failure modes ----
err = pd.DataFrame({"filename": train.filename, "true": y, "pred": oof_blend.round(2)})
err["abs_err"] = (err.pred - err.true).abs()
err = err.merge(tr_asr[["filename", "text", "n_words_asr"]], on="filename").sort_values("abs_err", ascending=False)
for _, r in err.head(6).iterrows():
    print(f"{r.filename}  true={r.true}  pred={r.pred}  words={r.n_words_asr}\n   {r.text[:220]}…\n")

## 9. Final metrics (training-set RMSE is required by the assessment)

In [ ]:
final = pd.DataFrame({
    "Out-of-fold (5-fold CV, honest estimate)": metrics(y, oof_blend),
    "In-sample on full training data":          metrics(y, ins_blend),
}).T.round(4)
print(final.to_string())
print(f"\n>>> TRAINING-DATA RMSE (in-sample)   : {final.iloc[1]['RMSE']:.4f}")
print(f">>> TRAINING-DATA RMSE (out-of-fold) : {final.iloc[0]['RMSE']:.4f}")
print(f">>> Pearson (out-of-fold)            : {final.iloc[0]['Pearson']:.4f}")

*Interpretation.* The in-sample numbers are optimistic (the models have seen those labels); the out-of-fold numbers are
what should be expected on the hidden test set. A large gap between the two would indicate over-fitting — the Ridge
regulariser, LightGBM early-stopping and the NNLS blend on OOF predictions are all chosen to keep that gap small.

## 10. Test predictions & submission

In [ ]:
# NOTE: sample_submission.csv shipped with the competition lists filenames that do not match test.csv
# (only 25 of its 204 rows appear in test.csv). The submission is therefore built from test.csv, in its order.
sub = pd.DataFrame({"filename": test.filename, "label": np.round(test_blend, 4)})
sub.to_csv("submission.csv", index=False)
print(sub.shape); sub.head(10)

In [ ]:
plt.figure(figsize=(6, 3.5))
sns.histplot(sub.label, bins=20, kde=True, color="#55A868")
plt.title("Predicted grammar scores on the test set"); plt.xlabel("score"); plt.tight_layout(); plt.show()

## 11. Conclusions & possible improvements

* **What works.** A transcript-first design lets a small, interpretable model do well: grammar-error density,
  dependency depth, fragment rate and ASR confidence are the dominant handcrafted signals, and the MiniLM embedding
  adds the "overall well-formedness" that counts alone miss. The 0-score clips are caught by the ASR-confidence /
  speech-ratio features rather than by a special-case rule.
* **Main error sources.** (1) Whisper itself silently *corrects* some grammatical errors and drops fillers, which
  compresses the signal for mid-range scores; (2) human ratings on a 0.5-step Likert scale are noisy, so adjacent bands
  (3 vs 3.5) are intrinsically hard; (3) a single 45–60 s sample contains only a handful of complex structures.
* **Next steps if more time / compute were available.**
  * Run `WHISPER_MODEL="large-v3"` and `RUN_DEBERTA=True` — in our experience the transformer fine-tune is the single
    strongest component and usually lifts OOF Pearson by several points.
  * Use a grammatical-error-correction model (e.g. T5-GEC) and count the edit distance between the raw and corrected
    transcript as a cleaner error measure than LanguageTool.
  * Add wav2vec2 / HuBERT pooled embeddings so that pronunciation-driven ASR failures are modelled directly.
  * Ordinal-aware losses or post-hoc rounding to the 0.5 grid, tuned on OOF RMSE.